In [ ]:
import os

# Create all folders
folders = [
    "indic_eval/models",
    "indic_eval/tasks",
    "indic_eval/metrics",
    "dashboard",
    "examples",
    "results",
    "data/samples",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Folder structure created!")

In [ ]:
!pip install openai sacrebleu rouge-score streamlit datasets -q
print("✅ Dependencies installed!")

In [ ]:
%%writefile indic_eval/__init__.py
"""
indic-eval: Evaluation harness for LLMs on Indic language tasks.
"""
__version__ = "0.1.0"
__author__ = "Shivam Kesarwani"

In [ ]:
%%writefile .gitignore
# Python
__pycache__/
*.py[cod]
*.pyo
*.pyd
.Python
*.egg-info/
dist/
build/
.eggs/

# Virtual environments
.venv/
venv/
env/

# Jupyter / Colab
.ipynb_checkpoints/
*.ipynb

# API keys — NEVER commit these
.env
*.env
secrets.py

# Results (too large for git, regenerate locally)
results/
*.json

# OS files
.DS_Store
Thumbs.db

# IDEs
.vscode/
.idea/

# HuggingFace model cache
.cache/

In [ ]:
%%writefile indic_eval/metrics/__init__.py
import re
from typing import List, Dict, Any
from dataclasses import dataclass, field

@dataclass
class MetricResult:
    name: str
    score: float
    details: Dict[str, Any] = field(default_factory=dict)

    def __repr__(self):
        return f"{self.name}: {self.score:.4f}"

def exact_match(predictions, references):
    def normalise(s):
        s = s.lower().strip()
        s = re.sub(r"[^\w\s]", "", s, flags=re.UNICODE)
        return s.strip()
    scores = [1.0 if normalise(p) == normalise(r) else 0.0
              for p, r in zip(predictions, references)]
    return MetricResult(
        name="exact_match",
        score=sum(scores) / len(scores) if scores else 0.0,
        details={"n_correct": int(sum(scores)), "n_total": len(scores)},
    )

def token_f1(predictions, references):
    def _f1(pred, ref):
        p_tok = pred.lower().split()
        r_tok = ref.lower().split()
        common = set(p_tok) & set(r_tok)
        if not common: return 0.0
        prec = len(common) / len(p_tok)
        rec  = len(common) / len(r_tok)
        return 2 * prec * rec / (prec + rec)
    scores = [_f1(p, r) for p, r in zip(predictions, references)]
    return MetricResult(name="token_f1",
                        score=sum(scores)/len(scores) if scores else 0.0)

def bleu(predictions, references):
    try:
        from sacrebleu.metrics import BLEU as SacreBLEU
        m = SacreBLEU(effective_order=True)
        r = m.corpus_score(predictions, [references])
        return MetricResult(name="bleu", score=r.score/100.0,
                            details={"sacrebleu": r.score})
    except Exception:
        scores = [len(set(p.lower().split()) & set(r.lower().split())) /
                  max(len(p.split()), 1) for p, r in zip(predictions, references)]
        return MetricResult(name="bleu", score=sum(scores)/len(scores) if scores else 0.0)

def chrf(predictions, references):
    try:
        from sacrebleu.metrics import CHRF
        m = CHRF()
        r = m.corpus_score(predictions, [references])
        return MetricResult(name="chrf", score=r.score/100.0)
    except Exception:
        def char_bigrams(s):
            return set(s[i:i+2] for i in range(len(s)-1))
        scores = []
        for p, r in zip(predictions, references):
            pg, rg = char_bigrams(p), char_bigrams(r)
            if not pg or not rg:
                scores.append(0.0); continue
            c = len(pg & rg)
            pr, rc = c/len(pg), c/len(rg)
            scores.append(2*pr*rc/(pr+rc) if (pr+rc) > 0 else 0.0)
        return MetricResult(name="chrf", score=sum(scores)/len(scores) if scores else 0.0)

def rouge_l(predictions, references):
    try:
        from rouge_score import rouge_scorer
        scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
        scores = [scorer.score(r, p)["rougeL"].fmeasure
                  for p, r in zip(predictions, references)]
    except Exception:
        def lcs_score(a, b):
            a, b = a.lower().split(), b.lower().split()
            m, n = len(a), len(b)
            dp = [[0]*(n+1) for _ in range(m+1)]
            for i in range(1, m+1):
                for j in range(1, n+1):
                    dp[i][j] = dp[i-1][j-1]+1 if a[i-1]==b[j-1] else max(dp[i-1][j], dp[i][j-1])
            l = dp[m][n]
            p_ = l/max(len(a), 1); r_ = l/max(len(b), 1)
            return 2*p_*r_/(p_+r_) if (p_+r_) > 0 else 0.0
        scores = [lcs_score(p, r) for p, r in zip(predictions, references)]
    return MetricResult(name="rouge_l", score=sum(scores)/len(scores) if scores else 0.0)

def mcq_accuracy(predictions, references):
    import re
    def extract(text):
        text = text.strip().upper()
        m = re.search(r"\b([ABCD])\b", text)
        return m.group(1) if m else text[:1]
    scores = [1.0 if extract(p)==extract(r) else 0.0
              for p, r in zip(predictions, references)]
    return MetricResult(name="mcq_accuracy",
                        score=sum(scores)/len(scores) if scores else 0.0,
                        details={"n_correct": int(sum(scores)), "n_total": len(scores)})

def latency_stats(latencies_ms):
    if not latencies_ms:
        return MetricResult(name="latency_ms", score=0.0)
    s = sorted(latencies_ms); n = len(s)
    return MetricResult(name="latency_ms", score=sum(s)/n,
                        details={"mean_ms": sum(s)/n,
                                 "p50_ms": s[n//2], "p95_ms": s[int(n*.95)]})

print("✅ Metrics module created!")

In [ ]:
%%writefile indic_eval/models/__init__.py
import os, time
from abc import ABC, abstractmethod
from typing import Optional
from dataclasses import dataclass

@dataclass
class ModelResponse:
    text: str
    latency_ms: float
    tokens_used: Optional[int] = None

class BaseModel(ABC):
    def __init__(self, model_name):
        self.model_name = model_name

    @abstractmethod
    def generate(self, prompt, max_tokens=256): pass

    def __repr__(self):
        return f"{self.__class__.__name__}(model={self.model_name})"

class APIModel(BaseModel):
    """OpenAI-compatible API — works with Sarvam AI, GPT, Ollama, etc."""
    def __init__(self, model_name, base_url="https://api.openai.com/v1",
                 api_key=None, temperature=0.0):
        super().__init__(model_name)
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key or os.environ.get("OPENAI_API_KEY", "")
        self.temperature = temperature

    def generate(self, prompt, max_tokens=256):
        from openai import OpenAI
        client = OpenAI(api_key=self.api_key, base_url=self.base_url)
        t0 = time.time()
        resp = client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens, temperature=self.temperature,
        )
        latency = (time.time() - t0) * 1000
        text = resp.choices[0].message.content or ""
        return ModelResponse(text=text.strip(), latency_ms=latency,
                             tokens_used=getattr(resp.usage, "total_tokens", None))

class HuggingFaceModel(BaseModel):
    """HuggingFace transformers — load any causal LM from the Hub."""
    def __init__(self, model_name, device="cpu", torch_dtype="auto", load_in_8bit=False):
        super().__init__(model_name)
        self.device = device
        self.torch_dtype = torch_dtype
        self.load_in_8bit = load_in_8bit
        self._pipeline = None

    def _load(self):
        if self._pipeline: return
        from transformers import pipeline
        self._pipeline = pipeline(
            "text-generation", model=self.model_name,
            device=self.device if not self.load_in_8bit else None,
            torch_dtype=self.torch_dtype, load_in_8bit=self.load_in_8bit,
        )

    def generate(self, prompt, max_tokens=256):
        self._load()
        t0 = time.time()
        out = self._pipeline(prompt, max_new_tokens=max_tokens,
                             do_sample=False, return_full_text=False)
        return ModelResponse(text=out[0]["generated_text"].strip(),
                             latency_ms=(time.time()-t0)*1000)

def load_model(config):
    t = config.get("type", "api").lower()
    if t == "api":
        return APIModel(config["model"], config.get("base_url","https://api.openai.com/v1"),
                        config.get("api_key"), config.get("temperature", 0.0))
    elif t in ("hf", "huggingface"):
        return HuggingFaceModel(config["model"], config.get("device","cpu"),
                                config.get("torch_dtype","auto"), config.get("load_in_8bit",False))
    raise ValueError(f"Unknown type: {t}")

print("✅ Models module created!")

In [ ]:
%%writefile indic_eval/tasks/__init__.py
from __future__ import annotations
import re, time
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import List, Dict, Any
from indic_eval.metrics import (exact_match, token_f1, bleu, chrf,
                                 rouge_l, mcq_accuracy, latency_stats, MetricResult)
from indic_eval.models import BaseModel, ModelResponse

@dataclass
class TaskResult:
    task_name: str
    metrics: List[MetricResult]
    predictions: List[str]
    references: List[str]
    latencies_ms: List[float]
    n_samples: int

    def primary_score(self):
        return self.metrics[0].score if self.metrics else 0.0

    def to_dict(self):
        return {"task": self.task_name, "n_samples": self.n_samples,
                "metrics": {m.name: round(m.score, 4) for m in self.metrics},
                "latency": {"mean_ms": round(sum(self.latencies_ms)/max(len(self.latencies_ms),1),1)}}

class BaseTask(ABC):
    name = "base"; description = ""; language = "hi"

    @abstractmethod
    def load_samples(self, n=50): pass

    @abstractmethod
    def parse_output(self, response): pass

    @abstractmethod
    def compute_metrics(self, predictions, references, latencies): pass

    def evaluate(self, model, n=50, verbose=False):
        samples = self.load_samples(n)
        predictions, references, latencies = [], [], []
        for i, s in enumerate(samples):
            if verbose: print(f"  [{i+1}/{len(samples)}] {self.name}", end="\r")
            resp = model.generate(s["prompt"], max_tokens=256)
            predictions.append(self.parse_output(resp))
            references.append(s["reference"])
            latencies.append(resp.latency_ms)
        metrics = self.compute_metrics(predictions, references, latencies)
        return TaskResult(self.name, metrics, predictions, references, latencies, len(samples))

# ── Task 1: Hindi Reading Comprehension ───────────────────────────
class HindiReadingComprehension(BaseTask):
    name = "hindi_reading_comprehension"
    description = "Reading comprehension in Hindi (IndicQA style)"
    language = "hi"
    PROMPT = """Read the following Hindi passage and answer the question.
Give only the answer, nothing else.

Passage: {context}

Question: {question}

Answer:"""

    def load_samples(self, n=50):
        try:
            from datasets import load_dataset
            ds = load_dataset("ai4bharat/IndicQA","IndicQA.hi",split="test",trust_remote_code=True)
            out = []
            for row in ds.select(range(min(n, len(ds)))):
                if row["answers"]["text"]:
                    out.append({"prompt": self.PROMPT.format(context=row["context"],
                                                              question=row["question"]),
                                "reference": row["answers"]["text"][0]})
            return out[:n]
        except Exception:
            return self._fallback(n)

    def _fallback(self, n):
        s = [
            {"prompt": self.PROMPT.format(
                context="महात्मा गांधी का जन्म 2 अक्टूबर 1869 को पोरबंदर, गुजरात में हुआ था।",
                question="महात्मा गांधी का जन्म कब हुआ था?"),
             "reference": "2 अक्टूबर 1869"},
            {"prompt": self.PROMPT.format(
                context="ताजमहल आगरा में स्थित है। इसे शाहजहाँ ने बनवाया था।",
                question="ताजमहल का निर्माण किसने करवाया?"),
             "reference": "शाहजहाँ"},
            {"prompt": self.PROMPT.format(
                context="भारत का राष्ट्रीय पक्षी मोर है। इसे 1963 में घोषित किया गया।",
                question="मोर को राष्ट्रीय पक्षी कब घोषित किया गया?"),
             "reference": "1963"},
        ]
        return (s * ((n // len(s)) + 1))[:n]

    def parse_output(self, r): return r.text.strip().split("\n")[0]
    def compute_metrics(self, p, r, l):
        return [exact_match(p, r), token_f1(p, r), latency_stats(l)]

# ── Task 2: English → Hindi Translation ───────────────────────────
class EnglishToHindiTranslation(BaseTask):
    name = "en_hi_translation"
    description = "English → Hindi translation (FLORES-200)"
    language = "hi"
    PROMPT = """Translate the following English sentence to Hindi.
Give only the Hindi translation, nothing else.

English: {source}

Hindi:"""

    def load_samples(self, n=50):
        try:
            from datasets import load_dataset
            ds = load_dataset("facebook/flores","eng_Latn-hin_Deva",split="devtest",trust_remote_code=True)
            return [{"prompt": self.PROMPT.format(source=row["sentence_eng_Latn"]),
                     "reference": row["sentence_hin_Deva"]}
                    for row in ds.select(range(min(n, len(ds))))]
        except Exception:
            return self._fallback(n)

    def _fallback(self, n):
        s = [
            {"prompt": self.PROMPT.format(source="India is a diverse country with many languages."),
             "reference": "भारत एक विविध देश है जिसमें कई भाषाएँ हैं।"},
            {"prompt": self.PROMPT.format(source="The train arrives at eight in the morning."),
             "reference": "ट्रेन सुबह आठ बजे पहुँचती है।"},
            {"prompt": self.PROMPT.format(source="She is studying artificial intelligence."),
             "reference": "वह कृत्रिम बुद्धिमत्ता का अध्ययन कर रही है।"},
        ]
        return (s * ((n // len(s)) + 1))[:n]

    def parse_output(self, r): return r.text.strip().split("\n")[0]
    def compute_metrics(self, p, r, l): return [bleu(p, r), chrf(p, r), latency_stats(l)]

# ── Task 3: Hinglish Sentiment ─────────────────────────────────────
class HinglishSentiment(BaseTask):
    name = "hinglish_sentiment"
    description = "Sentiment classification on Hinglish (code-switched) text"
    language = "hi-en"
    PROMPT = """Classify the sentiment of this Hinglish text as Positive, Negative, or Neutral.
Give only one word: Positive, Negative, or Neutral.

Text: {text}

Sentiment:"""
    SAMPLES = [
        ("Yaar ye movie bilkul bakwaas thi, time waste ho gaya.", "Negative"),
        ("Aaj ka din bahut achha tha! Sab kuch perfect raha.", "Positive"),
        ("Theek hai, kuch khaas nahi tha.", "Neutral"),
        ("Itna bura khana maine kabhi nahi khaya!", "Negative"),
        ("Bhai tera code bahut clean hai, mast kaam!", "Positive"),
        ("Train late thi as usual, koi surprise nahi.", "Negative"),
        ("Dekha nahi abhi, baad mein dekhenge.", "Neutral"),
        ("Wow yaar, ye jagah ekdum fantastic hai!", "Positive"),
        ("Service theek thi, na zyada acchi na buri.", "Neutral"),
        ("Phir se bijli gayi! Kab sudhrega ye system?", "Negative"),
    ]

    def load_samples(self, n=50):
        s = [{"prompt": self.PROMPT.format(text=t), "reference": lb} for t, lb in self.SAMPLES]
        return (s * ((n // len(s)) + 1))[:n]

    def parse_output(self, r):
        text = r.text.strip().split("\n")[0].capitalize()
        for lb in ["Positive", "Negative", "Neutral"]:
            if lb.lower() in text.lower(): return lb
        return text

    def compute_metrics(self, p, r, l): return [exact_match(p, r), latency_stats(l)]

# ── Task 4: Indian Cultural Reasoning ─────────────────────────────
class IndianCulturalReasoning(BaseTask):
    name = "indian_cultural_reasoning"
    description = "MCQ on Indian culture, history, geography, society"
    language = "hi"
    PROMPT = """Answer this multiple choice question about India. Give only the letter (A, B, C, or D).

Question: {question}
A) {a}
B) {b}
C) {c}
D) {d}

Answer:"""
    SAMPLES = [
        ("Which classical dance form originated in Kerala?",
         "Bharatanatyam","Kathakali","Odissi","Manipuri","B"),
        ("The festival of Onam is celebrated in which state?",
         "Tamil Nadu","Karnataka","Kerala","Andhra Pradesh","C"),
        ("Who wrote Jana Gana Mana?",
         "Bankim Chandra","Rabindranath Tagore","Sarojini Naidu","Subramania Bharati","B"),
        ("Which river is holiest in Hinduism?",
         "Yamuna","Saraswati","Ganga","Godavari","C"),
        ("Ajanta Caves are located in which state?",
         "Rajasthan","Madhya Pradesh","Maharashtra","Gujarat","C"),
        ("Pongal is the harvest festival of which community?",
         "Punjabis","Tamils","Bengalis","Gujaratis","B"),
        ("Largest state in India by area?",
         "Maharashtra","Uttar Pradesh","Madhya Pradesh","Rajasthan","D"),
        ("Indian Constitution was adopted on?",
         "15 Aug 1947","26 Jan 1950","26 Nov 1949","2 Oct 1950","C"),
        ("Which instrument is Ravi Shankar known for?",
         "Tabla","Sarod","Sitar","Veena","C"),
        ("Which city is famous for Biryani in Mughal history?",
         "Delhi","Agra","Hyderabad","Lucknow","C"),
    ]

    def load_samples(self, n=50):
        s = [{"prompt": self.PROMPT.format(question=q,a=a,b=b,c=c,d=d),
              "reference": ans} for q,a,b,c,d,ans in self.SAMPLES]
        return (s * ((n // len(s)) + 1))[:n]

    def parse_output(self, r): return r.text.strip()
    def compute_metrics(self, p, r, l): return [mcq_accuracy(p, r), latency_stats(l)]

# ── Task 5: Hindi Summarisation ────────────────────────────────────
class HindiSummarisation(BaseTask):
    name = "hindi_summarisation"
    description = "Abstractive summarisation of Hindi news text"
    language = "hi"
    PROMPT = """Summarise the following Hindi news article in 1-2 sentences.
Give only the summary in Hindi.

Article: {article}

Summary:"""
    SAMPLES = [
        ("नई दिल्ली: इसरो ने आज सफलतापूर्वक एक नया उपग्रह लॉन्च किया। यह उपग्रह संचार सेवाओं को बेहतर बनाएगा।",
         "इसरो ने एक नया संचार उपग्रह सफलतापूर्वक लॉन्च किया।"),
        ("मुंबई: वैश्विक कमजोरी के कारण सेंसेक्स में आज 500 अंकों की गिरावट आई। विशेषज्ञों का कहना है कि अनिश्चितता जारी रहेगी।",
         "वैश्विक कमजोरी से सेंसेक्स 500 अंक गिरा।"),
    ]

    def load_samples(self, n=50):
        s = [{"prompt": self.PROMPT.format(article=a), "reference": r} for a,r in self.SAMPLES]
        return (s * ((n // len(s)) + 1))[:n]

    def parse_output(self, r): return r.text.strip().split("\n")[0]
    def compute_metrics(self, p, r, l): return [rouge_l(p, r), chrf(p, r), latency_stats(l)]

# ── Registry ────────────────────────────────────────────────────────
TASK_REGISTRY = {
    "hindi_reading_comprehension": HindiReadingComprehension,
    "en_hi_translation":           EnglishToHindiTranslation,
    "hinglish_sentiment":          HinglishSentiment,
    "indian_cultural_reasoning":   IndianCulturalReasoning,
    "hindi_summarisation":         HindiSummarisation,
}
ALL_TASKS = list(TASK_REGISTRY.keys())

def get_task(name):
    if name not in TASK_REGISTRY:
        raise ValueError(f"Unknown task: '{name}'. Available: {ALL_TASKS}")
    return TASK_REGISTRY[name]()

def list_tasks():
    return [{"name": cls().name, "description": cls().description, "language": cls().language}
            for cls in TASK_REGISTRY.values()]

print("✅ Tasks module created!")

In [ ]:
%%writefile indic_eval/evaluator.py
import json, time
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Dict, Any
from indic_eval.models import BaseModel
from indic_eval.tasks import get_task, ALL_TASKS, TaskResult

@dataclass
class EvalReport:
    model_name: str
    tasks: List[TaskResult]
    total_time_s: float
    timestamp: str = field(default_factory=lambda: time.strftime("%Y-%m-%dT%H:%M:%S"))

    def overall_score(self):
        return sum(t.primary_score() for t in self.tasks) / len(self.tasks) if self.tasks else 0.0

    def summary(self):
        return {"model": self.model_name, "timestamp": self.timestamp,
                "total_time_s": round(self.total_time_s, 2),
                "tasks": [t.to_dict() for t in self.tasks],
                "overall_score": round(self.overall_score(), 4)}

    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.summary(), f, ensure_ascii=False, indent=2)
        print(f"💾 Results saved to {path}")

    def print_table(self):
        W = 36
        print(); print("="*70)
        print(f"  indic-eval  |  Model: {self.model_name}")
        print("="*70)
        print(f"  {'Task':<{W}} {'Metric':<20} {'Score':>8}")
        print("-"*70)
        for t in self.tasks:
            m = t.metrics[0]
            lat = next((x for x in t.metrics if x.name=="latency_ms"), None)
            lat_str = f"  ({lat.score:.0f}ms)" if lat else ""
            print(f"  {t.task_name:<{W}} {m.name:<20} {m.score*100:>7.1f}%{lat_str}")
        print("-"*70)
        print(f"  {'Overall Score':<{W}} {'(mean)':<20} {self.overall_score()*100:>7.1f}%")
        print("="*70)
        print(f"  Done in {self.total_time_s:.1f}s  |  {self.timestamp}"); print()

class Evaluator:
    def __init__(self, model, tasks=None, n_samples=50, verbose=True):
        self.model = model
        self.task_names = tasks or ALL_TASKS
        self.n_samples = n_samples
        self.verbose = verbose

    def run(self):
        results = []
        t0 = time.time()
        for name in self.task_names:
            task = get_task(name)
            if self.verbose: print(f"\n▶  {name}  ({self.n_samples} samples)")
            result = task.evaluate(self.model, n=self.n_samples, verbose=self.verbose)
            results.append(result)
            if self.verbose:
                m = result.metrics[0]
                print(f"   ✓ {m.name}: {m.score*100:.1f}%")
        report = EvalReport(self.model.model_name, results, time.time()-t0)
        if self.verbose: report.print_table()
        return report

print("✅ Evaluator created!")

In [ ]:
import sys, random
sys.path.insert(0, ".")

from indic_eval.models import BaseModel, ModelResponse
from indic_eval.evaluator import Evaluator

class MockModel(BaseModel):
    """Fake model that returns plausible answers — for testing only."""
    def generate(self, prompt, max_tokens=256):
        if "Passage:" in prompt:     answer = "2 अक्टूबर 1869"
        elif "Translate" in prompt:  answer = "भारत एक विविध देश है।"
        elif "Hinglish" in prompt:   answer = "Negative"
        elif "choice" in prompt:     answer = "C"
        else:                        answer = "इसरो ने उपग्रह लॉन्च किया।"
        return ModelResponse(text=answer, latency_ms=random.uniform(80, 200))

model = MockModel("mock-v1")
evaluator = Evaluator(model=model, n_samples=6, verbose=True)
report = evaluator.run()
report.save("results/mock_results.json")

In [ ]:
%%writefile .env
OPENAI_API_KEY=your_key_here
GROQ_API_KEY=your_key_here
SARVAM_API_KEY=your_key_here

In [ ]:
import os
from dotenv import load_dotenv
#os.environ["OPENAI_API_KEY"] = "****"  # paste your key

load_dotenv()



from indic_eval.models import APIModel
from indic_eval.evaluator import Evaluator

model = APIModel(
    model_name="llama-3.1-8b-instant",
    base_url="https://api.groq.com/openai/v1",
    api_key  = os.environ.get("OPENAI_API_KEY")
)

evaluator = Evaluator(
    model=model,
    tasks=["hinglish_sentiment", "indian_cultural_reasoning"],
    n_samples=10,
    verbose=True,
)
report = evaluator.run()
report.save("results/llama3_results.json")

In [ ]:
import os
from openai import OpenAI

# Assuming OPENAI_API_KEY is already set in the environment or notebook
# os.environ["OPENAI_API_KEY"] is already handled in previous cells.
api_key = os.environ.get("OPENAI_API_KEY")
base_url = "https://api.groq.com/openai/v1"

client = OpenAI(api_key=api_key, base_url=base_url)

try:
    models = client.models.list()
    print("Available Groq Models:")
    for model in models.data:
        print(f"- {model.id}")
except Exception as e:
    print(f"Error fetching models: {e}")